<a href="https://colab.research.google.com/github/abuhussein1504/NYC-Taxi-Trip-Duration/blob/main/NYC_Taxi_Trip_Duration_Modeling_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

TRAIN_PATH = r"/content/train.csv"
TEST_PATH = r"/content/test.csv"
SUBMISSION_OUTPUT_PATH = "submission.csv"

POLY_DEGREE = 2
RANDOM_STATE = 42


# --------------------------------------------------------------------------
# Feature engineering
# --------------------------------------------------------------------------

def extract_datetime_features(df, datetime_cols):
    df = df.copy()
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        prefix = col.replace("_datetime", "") if "_datetime" in col else col

        df[f"{prefix}_second"] = df[col].dt.second
        df[f"{prefix}_minute"] = df[col].dt.minute
        df[f"{prefix}_hour"] = df[col].dt.hour
        df[f"{prefix}_dayofweek"] = df[col].dt.dayofweek
        df[f"{prefix}_day"] = df[col].dt.day
        df[f"{prefix}_month"] = df[col].dt.month
        # df[f"{prefix}_year"] = df[col].dt.year
    return df


def haversine_miles(lat1, lat2, lon1, lon2):
    R = 3958.8
    lat1, lat2, lon1, lon2 = map(np.radians, [lat1, lat2, lon1, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return c * R


def engineer_features(df):
    df = extract_datetime_features(df, ["pickup_datetime"])
    df["distance"] = haversine_miles(
        df["pickup_latitude"], df["dropoff_latitude"],
        df["pickup_longitude"], df["dropoff_longitude"],
    )
    df["store_and_fwd_flag"] = (df["store_and_fwd_flag"] == "Y").astype(int)
    return df


def prepare_model_frame(df, drop_cols, fill_medians=None):
    df = df.drop(columns=drop_cols).drop_duplicates()
    medians = fill_medians if fill_medians is not None else df.median(numeric_only=True)
    df = df.fillna(medians)
    return df, medians


# --------------------------------------------------------------------------
# Load and engineer
# --------------------------------------------------------------------------

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)          # real Kaggle test set: NO trip_duration column

train = engineer_features(train_raw)
test = engineer_features(test_raw)

train_copy, train_medians = prepare_model_frame(
    train, drop_cols=["id", "pickup_datetime", "dropoff_datetime"]
)
# Test set never had dropoff_datetime (that would leak the target) or trip_duration
test_copy, _ = prepare_model_frame(
    test, drop_cols=["id", "pickup_datetime"], fill_medians=train_medians.drop("trip_duration")
)

# --------------------------------------------------------------------------
# Build feature matrix / target
# --------------------------------------------------------------------------

X = train_copy.drop(columns=["trip_duration"])
y = np.log1p(train_copy["trip_duration"])

X["distance"] = np.log1p(X["distance"])
test_copy["distance"] = np.log1p(test_copy["distance"])

# All labeled data comes from train.csv. Real test.csv has no labels, so it's
# held out separately and only used to produce the final submission.
X_train, X_valtest, y_train, y_valtest = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_valtest, y_valtest, test_size=0.5, random_state=RANDOM_STATE
)

# --------------------------------------------------------------------------
# Polynomial features - fit once on train, transform elsewhere
# --------------------------------------------------------------------------

poly = PolynomialFeatures(degree=POLY_DEGREE, include_bias=True)
X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)
X_test_poly = poly.transform(X_test)
X_submit_poly = poly.transform(test_copy) ###

# --------------------------------------------------------------------------
# Scaling - fit once on train, transform elsewhere
# --------------------------------------------------------------------------

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_val_scaled = scaler.transform(X_val_poly)
X_test_scaled = scaler.transform(X_test_poly)
X_submit_scaled = scaler.transform(X_submit_poly) ###
print("Scaler done.")

# --------------------------------------------------------------------------
# Ridge, tuned via RidgeCV
# --------------------------------------------------------------------------

ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge_cv.fit(X_train_scaled, y_train)
print("Best alpha:", ridge_cv.alpha_)

ridge_model = Ridge(alpha=ridge_cv.alpha_)
ridge_model.fit(X_train_scaled, y_train)

ridge_train_r2 = r2_score(y_train, ridge_model.predict(X_train_scaled)) # stop
ridge_val_r2 = r2_score(y_val, ridge_model.predict(X_val_scaled))
print(f"Ridge Train R²: {ridge_train_r2:.4f}")
print(f"Ridge Validation R²: {ridge_val_r2:.4f}")

cv_scores = cross_val_score(ridge_model, X_train_scaled, y_train, cv=5)
print("Ridge Cross-Validation Scores:", cv_scores)
print("Ridge Mean CV Score:", np.mean(cv_scores))

# --------------------------------------------------------------------------
# XGBoost, for comparison
# --------------------------------------------------------------------------

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)
xgb_model.fit(X_train_scaled, y_train)

xgb_train_r2 = r2_score(y_train, xgb_model.predict(X_train_scaled))
xgb_val_r2 = r2_score(y_val, xgb_model.predict(X_val_scaled))
print(f"XGB Train R²: {xgb_train_r2:.4f}")
print(f"XGB Validation R²: {xgb_val_r2:.4f}")

# --------------------------------------------------------------------------
# Final held-out check (only look at this once, after model selection)
# --------------------------------------------------------------------------

best_model = xgb_model if xgb_val_r2 >= ridge_val_r2 else ridge_model
best_name = "XGB" if best_model is xgb_model else "Ridge"
test_r2 = r2_score(y_test, best_model.predict(X_test_scaled))
print(f"\nSelected model: {best_name}")
print(f"{best_name} Held-out Test R²: {test_r2:.4f}")

# --------------------------------------------------------------------------
# Generate the actual Kaggle submission
# --------------------------------------------------------------------------

# submission_log_preds = best_model.predict(X_submit_scaled)
# submission_preds = np.expm1(submission_log_preds)  # back out of log space

# submission = pd.DataFrame({
#     "id": test_raw["id"],
#     "trip_duration": submission_preds,
# })
# submission.to_csv(SUBMISSION_OUTPUT_PATH, index=False)
# print(f"\nSubmission written to {SUBMISSION_OUTPUT_PATH}")

Scaler done.
Best alpha: 0.001
Ridge Train R²: 0.6324
Ridge Validation R²: 0.6007
Ridge Cross-Validation Scores: [0.63920274 0.63288749 0.62606062 0.6172155  0.62698973]
Ridge Mean CV Score: 0.628471214050793
XGB Train R²: 0.9488
XGB Validation R²: 0.6744

Selected model: XGB
XGB Held-out Test R²: 0.6841
